# Smart Grid Electricity Consumption Analysis and Forecasting

A reproducible notebook companion to the Streamlit dashboard. It downloads the official UCI archive, cleans the one-minute measurements, performs EDA and Review 3 checks, compares Linear Regression and Random Forest, and exports daily predictions.

**Target:** daily mean `Global_active_power` in kW. The dataset describes one household, not a utility-scale smart grid.

## 1. Install and import libraries

In [ ]:
%pip -q install pandas numpy matplotlib scipy scikit-learn requests

import io, zipfile, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
DATA_URL = "https://archive.ics.uci.edu/static/public/235/individual%2Bhousehold%2Belectric%2Bpower%2Bconsumption.zip"
MEASURE_COLUMNS = [
    "Global_active_power", "Global_reactive_power", "Voltage",
    "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"
]


## 2. Download and load the UCI dataset

In [ ]:
response = requests.get(DATA_URL, timeout=180)
response.raise_for_status()
with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    txt_name = next(name for name in archive.namelist() if name.lower().endswith(".txt"))
    txt_bytes = archive.read(txt_name)

raw = pd.read_csv(io.BytesIO(txt_bytes), sep=";", na_values="?", low_memory=False)
print("Shape:", raw.shape)
display(raw.head())
print("\nOriginal data types:")
display(raw.dtypes.to_frame("dtype"))
print("\nMissing values before cleaning:")
display(raw.isna().sum().to_frame("missing_count"))


## 3. Clean the data

In [ ]:
df = raw.copy()
df["Datetime"] = pd.to_datetime(
    df["Date"].astype(str) + " " + df["Time"].astype(str),
    dayfirst=True, errors="coerce"
)
df = df.dropna(subset=["Datetime"]).set_index("Datetime").sort_index()
df = df[~df.index.duplicated(keep="first")]
df = df.drop(columns=["Date", "Time"], errors="ignore")
for col in MEASURE_COLUMNS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

missing_before = int(df[MEASURE_COLUMNS].isna().sum().sum())
df[MEASURE_COLUMNS] = df[MEASURE_COLUMNS].interpolate(method="time").ffill().bfill()
missing_after = int(df[MEASURE_COLUMNS].isna().sum().sum())
print(f"Missing measurement values before: {missing_before:,}")
print(f"Missing measurement values after:  {missing_after:,}")
print("Cleaned dtypes:")
display(df[MEASURE_COLUMNS].dtypes.to_frame("dtype"))


## 4. Exploratory data analysis

In [ ]:
daily = df["Global_active_power"].resample("D").mean().dropna().rename("Power")
fig, ax = plt.subplots(figsize=(14, 4))
daily.plot(ax=ax, color="#167D73", linewidth=1)
ax.set(title="Daily Mean Global Active Power", xlabel="Date", ylabel="Power (kW)")
ax.grid(alpha=.25); plt.show()

hourly = df.groupby(df.index.hour)["Global_active_power"].mean()
fig, ax = plt.subplots(figsize=(10, 4))
hourly.plot(kind="bar", ax=ax, color="#73A9B8")
ax.set(title="Average Active Power by Hour", xlabel="Hour of Day", ylabel="Power (kW)")
plt.tight_layout(); plt.show()

weekly = daily.resample("W").mean()
monthly = df["Global_active_power"].resample("MS").mean()
fig, ax = plt.subplots(figsize=(13, 4)); weekly.plot(ax=ax, color="#D89A3D")
ax.set(title="Weekly Average Active Power", ylabel="Power (kW)"); ax.grid(alpha=.25); plt.show()
fig, ax = plt.subplots(figsize=(13, 4)); monthly.plot(ax=ax, color="#2876A5")
ax.set(title="Monthly Average Active Power", ylabel="Power (kW)"); ax.grid(alpha=.25); plt.show()

correlation = df[MEASURE_COLUMNS].corr()
fig, ax = plt.subplots(figsize=(9, 7))
image = ax.imshow(correlation, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(MEASURE_COLUMNS)), MEASURE_COLUMNS, rotation=90)
ax.set_yticks(range(len(MEASURE_COLUMNS)), MEASURE_COLUMNS)
for i in range(len(MEASURE_COLUMNS)):
    for j in range(len(MEASURE_COLUMNS)):
        ax.text(j, i, f"{correlation.iloc[i,j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(image, ax=ax, label="Pearson correlation")
ax.set_title("Correlation Heatmap of Electrical Measurements")
plt.tight_layout(); plt.show()


## 5. Normality, outlier and scaling checks

In [ ]:
power = df["Global_active_power"].dropna()
sample = power.sample(n=min(5000, len(power)), random_state=RANDOM_STATE)
shapiro_stat, shapiro_p = stats.shapiro(sample)
dagostino_stat, dagostino_p = stats.normaltest(sample)
q1, q3 = power.quantile([.25, .75]); iqr = q3 - q1
lower_bound, upper_bound = q1 - 1.5*iqr, q3 + 1.5*iqr
outlier_mask = (power < lower_bound) | (power > upper_bound)

print(f"Shapiro-Wilk statistic={shapiro_stat:.5f}, p-value={shapiro_p:.6g}")
print(f"D'Agostino statistic={dagostino_stat:.5f}, p-value={dagostino_p:.6g}")
print(f"Skewness={power.skew():.5f}")
print(f"Excess kurtosis={power.kurtosis():.5f}")
print(f"IQR lower={lower_bound:.5f}, upper={upper_bound:.5f}")
print(f"IQR outlier count={int(outlier_mask.sum()):,} ({outlier_mask.mean()*100:.3f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(power, bins=60, color="#81B4C5", edgecolor="white", density=True)
axes[0].set(title="Normality Check: Global Active Power", xlabel="Power (kW)", ylabel="Density")
stats.probplot(sample, dist="norm", plot=axes[1])
axes[1].set_title("Normal Q-Q Plot")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(12, 3))
ax.boxplot(power, vert=False, flierprops={"marker":"o", "markersize":3})
ax.axvline(lower_bound, color="crimson", linestyle="--", label="IQR lower bound")
ax.axvline(upper_bound, color="crimson", linestyle="--", label="IQR upper bound")
ax.set(title="IQR Outlier Check: Global Active Power", xlabel="Power (kW)")
ax.legend(); plt.tight_layout(); plt.show()

scale_sample = power.sample(n=min(12000, len(power)), random_state=RANDOM_STATE).to_numpy().reshape(-1,1)
standardised = StandardScaler().fit_transform(scale_sample).ravel()
normalised = MinMaxScaler().fit_transform(scale_sample).ravel()
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, values, title in zip(axes, [scale_sample.ravel(), standardised, normalised], ["Original", "Standardised (z-score)", "Min-Max normalised"]):
    ax.hist(values, bins=45, color="#66A6B8", edgecolor="white")
    ax.set_title(title); ax.set_ylabel("Count")
plt.suptitle("Scaling Comparison: Global Active Power"); plt.tight_layout(); plt.show()

print("Interpretation: IQR flags are candidates, not automatic errors. Keep real load peaks unless investigated as data faults.")


## 6. Feature engineering and chronological model evaluation

In [ ]:
supervised = pd.DataFrame({"Power": daily})
supervised["Lag_1"] = supervised["Power"].shift(1)
supervised["Lag_7"] = supervised["Power"].shift(7)
supervised["Lag_14"] = supervised["Power"].shift(14)
supervised["Rolling_7"] = supervised["Power"].shift(1).rolling(7).mean()
supervised["Day"] = supervised.index.day
supervised["Month"] = supervised.index.month
supervised["DayOfWeek"] = supervised.index.dayofweek
supervised = supervised.dropna()

features = ["Lag_1", "Lag_7", "Lag_14", "Rolling_7", "Day", "Month", "DayOfWeek"]
split = int(len(supervised) * 0.8)
X_train, X_test = supervised[features].iloc[:split], supervised[features].iloc[split:]
y_train, y_test = supervised["Power"].iloc[:split], supervised["Power"].iloc[split:]

models = {
    "Linear Regression": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
}
rows, prediction_frame = [], pd.DataFrame(index=y_test.index)
prediction_frame["Actual"] = y_test
for name, model in models.items():
    model.fit(X_train, y_train)
    predicted = model.predict(X_test)
    prediction_frame[name + " Predicted"] = predicted
    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predicted),
        "RMSE": np.sqrt(mean_squared_error(y_test, predicted)),
        "R2": r2_score(y_test, predicted),
    })
metrics = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
best_model_name = metrics.loc[0, "Model"]
best_model = models[best_model_name]
display(metrics.style.format({"MAE":"{:.4f}", "RMSE":"{:.4f}", "R2":"{:.4f}"}))
print("Best model by holdout RMSE:", best_model_name)

fig, ax = plt.subplots(figsize=(13, 4))
prediction_frame.plot(ax=ax, linewidth=1.2)
ax.set(title="Actual vs Predicted Daily Mean Active Power", ylabel="Power (kW)")
ax.grid(alpha=.25); plt.tight_layout(); plt.show()


## 7. Recursive forecast and CSV export

In [ ]:
horizon = 14
history = list(daily.astype(float).values)
last_date = daily.index[-1]
future_rows = []
for step in range(1, horizon + 1):
    future_date = last_date + pd.Timedelta(days=step)
    features_row = pd.DataFrame([{
        "Lag_1": history[-1], "Lag_7": history[-7], "Lag_14": history[-14],
        "Rolling_7": float(np.mean(history[-7:])), "Day": future_date.day,
        "Month": future_date.month, "DayOfWeek": future_date.dayofweek,
    }], index=[future_date])[features]
    value = max(0.0, float(best_model.predict(features_row)[0]))
    history.append(value)
    future_rows.append({"Date": future_date, "Forecast_kW": value})
future = pd.DataFrame(future_rows).set_index("Date")

fig, ax = plt.subplots(figsize=(13, 4))
daily.tail(120).plot(ax=ax, label="Historical", color="#243B53")
future["Forecast_kW"].plot(ax=ax, label="Forecast", color="#0E8A74", style="--", marker="o")
ax.set(title=f"Next {horizon} Days — {best_model_name}", ylabel="Daily mean active power (kW)")
ax.legend(); ax.grid(alpha=.25); plt.tight_layout(); plt.show()

prediction_frame.to_csv("daily_electricity_forecasting.csv", index_label="Date")
metrics.to_csv("smart_grid_model_metrics.csv", index=False)
future.to_csv("smart_grid_future_forecast.csv")
print("Saved: daily_electricity_forecasting.csv, smart_grid_model_metrics.csv, smart_grid_future_forecast.csv")


## 8. Conclusion and interpretation notes

- Compare both models using the chronological holdout scores; do not describe either as highly accurate unless the metrics support that conclusion.
- Shapiro–Wilk and D’Agostino tests assess the distribution of sampled minute-level power readings. Normality of predictors is not a requirement for Random Forest; for Linear Regression, residual diagnostics are more relevant than requiring normally distributed predictors.
- Scaling is part of the Linear Regression pipeline and is fit only on training rows. Random Forest is evaluated without scaling.
- IQR outlier points are not automatically removed because genuine consumption peaks may be meaningful.
- This analysis uses one historical household; adding weather, occupancy, tariff and live smart-meter data would be future work.